# Solar Active-Region Detection — Kaggle training run (MAX-QUALITY preset)

**One cell to run.** Uses Kaggle's session Python directly (no venv — venv is
broken on Kaggle), downloads the data, and trains on **both** T4 GPUs.

Right-hand panel (**Session options**), before/while running:

1. **Internet: ON** — required for the download.
2. **GPU: T4 x2** — both are used automatically (DataParallel).
3. **Persistence: Files only** — WITHOUT this, every 12-hour session end WIPES
   your data + model. Set it while a session is running for it to apply.

Preset (T4x2 runtime): `BASE_CHANNELS=48 DEEP_SUPERVISION=1` (best-quality 18M
model, auto batch fits VRAM), 16 download workers, 2,000 frames, every core.

When a session ends (12 h): run this same cell again in the same notebook — it
resumes. The newest model is mirrored to /kaggle/output (Output tab) every 5 min.

In [ ]:
%%bashset -xexport HOME=/kaggle/workingcd /kaggle/working# Clean up artifacts from the broken first attempt (wrong dir + broken venv):rm -rf /kaggle/workif [ -d SOALR/.git ]; then    git -C SOALR pull -q || true    rm -rf SOALR/.venv   # venvs are broken on Kaggle; we use the session Pythonelse    git clone -q -b arena/01a04247-soalr-active-region-detection \        https://github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-.git SOALR \    || { mkdir -p SOALR && wget -qO /tmp/repo.tgz \        https://codeload.github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-/tar.gz/refs/heads/arena/01a04247-soalr-active-region-detection \        && tar xzf /tmp/repo.tgz -C SOALR --strip-components=1; }fiif [ ! -x SOALR/scripts/run_forever.sh ]; then    echo "STOP: repository download failed. In the notebook Settings, make sure Internet is ON, then re-run this cell."    exit 1ficd SOALR# NO venv on Kaggle (ensurepip is broken there). Install the small missing# packages straight into the session environment; torch is already installed.python3 -m pip install -q -r requirements.txtpython3 -c "import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())"# Kaggle only lets you DOWNLOAD files from /kaggle/output (zip available in the# notebook's "Output" tab after each session ends). This watcher keeps a fresh# copy of the latest model + log there every 5 minutes:mkdir -p /kaggle/output(    while :; do        sleep 300        cp -f /kaggle/working/solar_results/arpil/continuous/best.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/continuous/last.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/continuous/metrics.jsonl /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/run_forever.log      /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/STATUS.md            /kaggle/output/ 2>/dev/null    done) &SAVE_WATCHER=$!# MAX-QUALITY preset for Kaggle:#   SOLAR_PYTHON=python3   use the session Python (no venv on Kaggle)#   BASE_CHANNELS=48 + DEEP_SUPERVISION=1  best-quality model (both T4s via#                                          DataParallel, batch auto-fits VRAM)#   DOWNLOAD_WORKERS=16    saturates the download link#   MAX_TOTAL_FRAMES=2000  same dataset as the Mac run#   MIN_FREE_GB=4          disk safety reserve#   CPU_HEADROOM=0         every core to trainingCHANNELS="aia94 aia131 aia1600 aia171 aia193 aia211 aia304 aia335 hmi_m hmi_bx hmi_by hmi_bz hmi_v" \SOLAR_PYTHON="$(command -v python3)" BASE_CHANNELS=48 DEEP_SUPERVISION=1 \MIN_FREE_GB=4 MAX_TOTAL_FRAMES=2000 FRAMES_PER_CYCLE=200 DOWNLOAD_WORKERS=16 \CPU_HEADROOM=0 TILES_PER_EPOCH=200 VAL_EPOCH=10 VAL_SUBSET=300 \bash scripts/run_forever.shkill "$SAVE_WATCHER" 2>/dev/null

## What you will see

- `torch 2.x.x | CUDA available: True | GPUs: 2` — environment healthy
- Preflight `3 ok`, then `Downloading 200 frames with 16 parallel workers ...`
- `[stream] using 2 GPUs with DataParallel (each batch splits across both)`
- `[stream] epoch=0001 loss=0.7x ...` — finite loss; first `val_dice` at epoch 10

## If the session dies (12-hour cap or a crash)

Run this same cell again — same notebook. With **Persistence: Files only** set,
nothing is lost; it resumes downloads and training from the last checkpoint.